In [0]:
%sql
create schema if not exists workspace.silver

In [0]:
#Nessas células eu fiz o import e a leitura dos dados da tabela de customers na camada bronze.
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col, upper, row_number, lit, when, to_timestamp, date_diff, to_date, min, max, sequence, explode, last, round, sum

df_customers_bronze = spark.table("workspace.bronze.tb_customers")
display(df_customers_bronze)
df_customers_bronze.printSchema()

In [0]:
# Nessa celula eu vou renomear e padronizar o que eu li da tabela customers na camada bronze

df_consumidores = (
    df_customers_bronze
    .select(
        col("customer_id").alias("id_consumidor"),
        col("customer_zip_code_prefix").cast("int").alias("prefixo_cep"),
        col("customer_name").cast("string").alias("nome_consumidor"),
        upper(col("customer_city")).alias("cidade"),
        upper(col("customer_state")).alias("estado"),
        col("timestamp_ingestion")
    )
)

In [0]:
#aqui eu irei deduplicar 

janela = Window.partitionBy("id_consumidor").orderBy(col("timestamp_ingestion").desc())

df_consumidores = (
    df_consumidores
    .withColumn("rn",row_number().over(janela))
    .filter(col("rn")==1)
    .drop("rn")
)

In [0]:
# Agora vou gravar na silver esses dados tratados
df_consumidores.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_consumidores")

A partir de agora vou fazer tudo de uma tabela em um unica celula

In [0]:
# Vamos tratar agora a tabela de vendedores.
#para começar vamos ler a tabala da camada broze

df_sellers_bronze = spark.table("workspace.bronze.tb_sellers")
display(df_sellers_bronze)
df_sellers_bronze.printSchema()

#Lido, vamos agora renomear e aplicar as padronizações necessarias

df_vendedores = (
  df_sellers_bronze
  .select(
    col("seller_id").alias("id_vendedor"),
    col("seller_name").cast("string").alias("nome_vendedor"),
    col("seller_zip_code_prefix").cast("int").alias("prefixo_cep"),
    upper(col("seller_city")).alias("cidade"),
    upper(col("seller_state")).alias("estado"),
    col("timestamp_ingestion")
  )
)

#vamos agora fazer a deduplicação por time ingestion
janela = Window.partitionBy("id_vendedor").orderBy(col("timestamp_ingestion").desc())

df_vendedores = (
  df_vendedores
  .withColumn("rn", row_number().over(janela))
  .filter(col("rn")==1)
  .drop("rn")
)
#Vamos agora gravar na camada silver depois de fazermos as alteraçõe snecessarias

df_vendedores.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_vendedores")


In [0]:
#agora vamos ler produtos da camada bronze

df_products_bronze = spark.table("workspace.bronze.tb_products")
display(df_products_bronze)
df_products_bronze.printSchema()

# agora vamos aplicar a pradonização 

df_produtos = (
  df_products_bronze
  .select(
    col("product_id").alias("id_produto"),
    col("product_name").cast("string").alias("nome_produto"),
    col("product_category_name").alias("categoria_produto"),
    col("product_weight_g").cast("int").alias("peso_produto_gramas"),
    col("product_length_cm").cast("int").alias("comprimento_centimetros"),
    col("product_height_cm").cast("int").alias("altura_centimetros"),
    col("product_width_cm").cast("int").alias("largura_centimetros"),
    col("product_photos_qty").cast("int").alias("quantidade_fotos"),
    col("product_name_lenght").cast("int").alias("tamanho_nome_produto"),
    col("product_description_lenght").cast("int").alias("tamanho_descricao_produto"),
    col("timestamp_ingestion")
  )
)

# depois de aplicar tudo, vamos deduplicar

janela = Window.partitionBy("id_produto").orderBy(col("timestamp_ingestion").desc())

df_produtos = (
  df_produtos
  .withColumn("rn", row_number().over(janela))
  .filter(col("rn")==1)
  .drop("rn")
)

#agora gravar na silver

df_produtos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_produtos")

In [0]:
#leremos categoria produtos da bronze

df_category_products_bronze = spark.table("workspace.bronze.tb_product_category_name_translation")
display(df_category_products_bronze)
df_category_products_bronze.printSchema()

#aplicar padronização

df_categoria_produtos_traducao = (
    df_category_products_bronze
    .select(
        col("product_category_name").alias("nome_produto_pt"),
        col("product_category_name_english").alias("nome_produto_en")
    )
)

#agora é gravar na silver

df_categoria_produtos_traducao.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.silver.dim_categoria_produtos_traducao")

In [0]:
#agora vamos para order items 

df_order_items_bronze = spark.table("workspace.bronze.tb_order_items")
display(df_order_items_bronze)
df_order_items_bronze.printSchema()

#aplicar padronização

df_vendas = (
    df_order_items_bronze
    .select(
        col("order_id").alias("id_pedido"),
        col("order_item_id").alias("id_item"),
        col("product_id").alias("id_produto"),
        col("seller_id").alias("id_vendedor"),
        col("price").cast("decimal(10,2)").alias("preco_BRL"),
        col("freight_value").cast("decimal(10,2)").alias("preco_frete")
    )
)

#salvar na silver
df_vendas.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.silver.fat_itens_pedidos")

In [0]:
#para order payments
df_payments_bronze = spark.table("workspace.bronze.tb_order_payments")
display(df_payments_bronze)
df_payments_bronze.printSchema()

df_pagamentos_pedidos = (
    df_payments_bronze
    .select(
        col("order_id").alias("id_pedido"),
        when(col("payment_type") == "credit_card", "Cartão de Crédito")
        .when(col("payment_type") == "boleto", "Boleto")
        .when(col("payment_type") == "voucher", "Voucher")
        .when(col("payment_type") == "debit_card", "Cartão de Débito")
        .when(col("payment_type") == "not_defined", "Não Definido")
        .otherwise(col("payment_type")).alias("tipo_pagamento"),
        col("payment_value").cast("decimal(10,2)").alias("valor_pagamento"),
        col("payment_installments").cast("int").alias("qtd_parcelas")
    )
)

df_pagamentos_pedidos.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.silver.fat_pagamentos_pedidos")

In [0]:
#para pedidos
df_pedidos_bronze = spark.table("workspace.bronze.tb_orders")
display(df_pedidos_bronze)
df_pedidos_bronze.printSchema()

df_pedidos = (
    df_pedidos_bronze
    .select(
         col("order_id").alias("id_pedido"),
        col("customer_id").alias("id_consumidor"),

        when(col("order_status") == "delivered", "entregue")
        .when(col("order_status") == "canceled", "cancelado")
        .when(col("order_status") == "shipped", "enviado")
        .when(col("order_status") == "processing", "processando")
        .when(col("order_status") == "invoiced", "faturado")
        .when(col("order_status") == "unavailable", "indisponível")
        .when(col("order_status") == "created", "criado")
        .when(col("order_status") == "approved", "aprovado")
        .otherwise(col("order_status"))
        .alias("status"),

        to_timestamp(col("order_purchase_timestamp")).alias("data_pedido"),
        to_timestamp(col("order_approved_at")).alias("data_aprovacao"),
        to_timestamp(col("order_delivered_carrier_date")).alias("data_envio_transportadora"),
        to_timestamp(col("order_delivered_customer_date")).alias("data_entrega"),
        to_timestamp(col("order_estimated_delivery_date")).alias("data_estimada_entrega")
    )
)
# aqui eu fiz a criação e alogica de cada nova coluna 

df_pedidos = (
    df_pedidos
    .withColumn("tempo_entrega_dias", date_diff(col("data_entrega"),col("data_pedido")))
    .withColumn("tempo_entrega_estimado_dias", date_diff(col("data_estimada_entrega"), col("data_pedido")))
    .withColumn("diferenca_entrega_dias", col("tempo_entrega_dias") - col("tempo_entrega_estimado_dias"))
    .withColumn("entrega_no_prazo",
                when(
                    col("status") == "entregue",
                    when(col("tempo_entrega_dias") <= col("tempo_entrega_estimado_dias"), "Sim")
                    .otherwise("Não")).otherwise("Não Entregue")
                )
    )

df_pedidos.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("workspace.silver.fat_pedidos")

In [0]:
#para a avaliação de cada pedido
df_reviews = spark.table("workspace.bronze.tb_order_reviews")
df_orders_validos = spark.table("workspace.bronze.tb_orders").select("order_id").distinct()

# converter datas com tolerância a falhas
df_reviews = (
    df_reviews
    .withColumn(
        "review_creation_ts",
        F.expr("try_to_timestamp(review_creation_date)")
    )
    .withColumn(
        "review_answer_ts",
        F.expr("try_to_timestamp(review_answer_timestamp)")
    )
)

# manter apenas pedidos válidos
# left_semi join = só mantém linhas de reviews que têm order_id existente em orders
df_reviews = df_reviews.join(df_orders_validos, on="order_id", how="left_semi")

# remover comentários no futuro
df_reviews = df_reviews.filter(
    F.col("review_creation_ts").isNotNull() &
    (F.to_date("review_creation_ts") <= F.current_date())
)

# tratar nulos e strings vazias
df_reviews = (
    df_reviews
    .withColumn(
        "review_comment_title",
        F.when(
            F.col("review_comment_title").isNull() |
            (F.trim(F.col("review_comment_title")) == ""),
            F.lit("Sem título")
        ).otherwise(F.col("review_comment_title"))
    )
    .withColumn(
        "review_comment_message",
        F.when(
            F.col("review_comment_message").isNull() |
            (F.trim(F.col("review_comment_message")) == ""),
            F.lit("Sem comentário")
        ).otherwise(F.col("review_comment_message"))
    )
)

# selecionar e renomear colunas finais
df_silver_avaliacoes = (
    df_reviews.select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.col("review_score").alias("nota_avaliacao"),

        F.col("review_comment_title").alias("titulo_avaliacao"),
        F.col("review_comment_message").alias("comentario_avaliacao"),

        F.col("review_creation_ts").alias("data_criacao_avaliacao"),
        F.col("review_answer_ts").alias("data_resposta_avaliacao")
    )
)

display(df_silver_avaliacoes)
df_silver_avaliacoes.printSchema()

#salvar 
df_silver_avaliacoes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.fat_avaliacoes_pedidos")

In [0]:
# para a cotação do dolar
df_cotacao_dolar = spark.table("workspace.bronze.tb_cotacao_dolar")
df_cotacao = df_cotacao_dolar.select(
    to_date(col("dataHoraCotacao")).alias("data_cotacao"),
    col("dataHoraCotacao"),
    col("cotacaoCompra").cast("double").alias("cotacao_dolar_brl")
    )

# aplico a deduplicação para nao ter duas cotações e so pegar a ultima do dia

janela_dia = Window.partitionBy("data_cotacao").orderBy(col("dataHoraCotacao").desc())

df_cotacao_dia = (
    df_cotacao
    .withColumn("rn", row_number().over(janela_dia))
    .filter(col("rn") == 1)
    .drop("rn", "dataHoraCotacao")
)

# agora eu vou criar o calendario continuo

limites = df_cotacao_dia.select(
    min("data_cotacao").alias("data_min"),
    max("data_cotacao").alias("data_max")
)

df_calendario = limites.select(
    explode(sequence(col("data_min"), col("data_max"))).alias("data_cotacao")
)

#aqui eu vou juntar o calendario com a cotação
df_join = (
    df_calendario
    .join(df_cotacao_dia, on="data_cotacao", how="left")
    .orderBy("data_cotacao")
)

#preencho com a ultima cotação 
janela_preenchimento = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cotacao_preenchida = (
    df_join
    .withColumn(
        "cotacao_dolar_brl_preenchida",
        last("cotacao_dolar_brl", ignorenulls=True).over(janela_preenchimento)
    )
)

#escolhi o schema da tabela
df_dim_cotacao_dolar = (
    df_cotacao_preenchida
    .select(
        col("data_cotacao"),
        round(col("cotacao_dolar_brl_preenchida"), 4).alias("cotacao_dolar_brl")
    )
)

df_dim_cotacao_dolar.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.dim_cotacao_dolar")

In [0]:
#para pedido total
df_pedidos = spark.table("workspace.silver.fat_pedidos")
df_pagamentos = spark.table("workspace.silver.fat_pagamentos_pedidos")
df_cotacao = spark.table("workspace.silver.dim_cotacao_dolar")

#agrega os pagamentos por pedido
df_pagamentos_agg = (
    df_pagamentos
    .groupBy("id_pedido")
    .agg(
        sum("valor_pagamento").alias("valor_total_pago_brl")
    )
)
#prepara a data do pedido para juntar com a cotaçaõ
df_pedidos_base = df_pedidos.withColumn("data_pedido_join", to_date(col("data_pedido")))

#faz os joins de tabela
df_pedido_total = (
    df_pedidos_base
    .join(df_pagamentos_agg, on="id_pedido", how="left")
    .join(
        df_cotacao,
        df_pedidos_base["data_pedido_join"] == df_cotacao["data_cotacao"],
        how="left"
    )
)
#calcula e seleciona as colunas finais
df_pedido_total_final = (
    df_pedido_total
    .select(
        col("id_pedido"),
        col("id_consumidor"),
        col("status"),
        round(col("valor_total_pago_brl"), 2).alias("valor_total_pago_brl"),
        round(col("valor_total_pago_brl") / col("cotacao_dolar_brl"), 2).alias("valor_total_pago_usd"),
        col("data_pedido")
    )
)

display(df_pedido_total_final)
df_pedido_total_final.printSchema()

df_pedido_total_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.fat_pedido_total")

AGORA vou fazer o optimize para reorganizar os aquivos da tabela deltae o zorder by para agrupar os dados fisicamente 

In [0]:
%sql
OPTIMIZE workspace.silver.fat_pedidos
ZORDER BY (id_pedido, data_pedido);

OPTIMIZE workspace.silver.fat_pedido_total
ZORDER BY (id_pedido, data_pedido);

OPTIMIZE workspace.silver.fat_itens_pedidos
ZORDER BY (id_pedido);